# 🤖 Tarea 1 · Parte 2 — Del LLM al clasificador

**Flujo:** línea base zero-shot → etiquetado débil con un LLM local (Ollama) → auditoría
humana de 20 casos → split 70/15/15 → fine-tuning de un Transformer → evaluación en test →
comparativa y decisión.

## Decisión de diseño, explícita

El enunciado deja una ambigüedad importante: hay **dos etiquetas posibles** para entrenar.

| Columna | De dónde viene | Qué es |
|---|---|---|
| `etiqueta` | Sección del diario (notebook 1) | Referencia independiente, pero ruidosa: una nota sobre el presupuesto puede ir en `politica` en un diario y en `economia` en el otro |
| `etiqueta_llm` | LLM leyendo titular + bajada | La supervisión débil que pide el paso 2 |

**Se entrena con `etiqueta_llm`**, porque el enunciado pide explícitamente supervisión débil
con LLM. El test se evalúa contra `etiqueta_llm` (la métrica que va en el comentario de
entrega) y **además** contra `etiqueta`, que al no haber participado del entrenamiento
funciona como control externo. La brecha entre ambas cifras es uno de los hallazgos del
trabajo.

> 💡 Si tu laptop va lenta en el paso 5, este notebook corre igual en **Google Colab con GPU
> T4**: sube `datos/noticias.csv`, ejecútalo y descarga `modelo/` y `modelo/metricas.json`.

In [ ]:
import json, os, platform, random, re, subprocess, time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests

SEED = 42
random.seed(SEED); np.random.seed(SEED)

BASE   = Path.cwd()
DATOS  = BASE / "datos"
MODELO = BASE / "modelo"; MODELO.mkdir(exist_ok=True)

CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
ID2LAB = dict(enumerate(CATEGORIAS))
LAB2ID = {v: k for k, v in ID2LAB.items()}

df = pd.read_csv(DATOS / "noticias.csv").fillna({"bajada": ""})
print("Dataset:", df.shape)
display(df["etiqueta"].value_counts())

# texto que ve el modelo: titular + bajada, que es mas senal que el titular solo
df["texto"] = (df["titular"].str.strip() + ". " + df["bajada"].str.strip()).str.strip(". ")

## 0 · Detección de hardware

El profesor insistió en esto: por defecto todo cae a CPU y hay que decirle explícitamente
al código que use la GPU. Esta celda detecta **CUDA** (NVIDIA), **MPS** (Mac con Apple
Silicon) o CPU, y el resto del notebook usa ese dispositivo.

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE, DEVICE_ID = "cuda", 0
    detalle = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    detalle += f" ({vram:.1f} GB VRAM)"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE, DEVICE_ID = "mps", "mps"
    detalle = "Apple Silicon (Metal Performance Shaders)"
else:
    DEVICE, DEVICE_ID = "cpu", -1
    detalle = platform.processor() or "CPU"

print(f"Dispositivo : {DEVICE}")
print(f"Detalle     : {detalle}")
print(f"PyTorch     : {torch.__version__}")
if DEVICE == "cpu":
    print("\n  Aviso: sin GPU el fine-tuning toma entre 20 y 40 minutos con este dataset.")

## 1 · Línea base Zero-Shot (sin fine-tuning)

Es el **piso mínimo a superar**. Se evalúa sobre 40 titulares tomados al azar, de forma
estratificada para que las 4 categorías estén representadas.

En este punto del flujo las etiquetas del LLM todavía no existen, así que la referencia es
`etiqueta` (la sección del diario). En la tabla comparativa del paso 6 se vuelve a medir
todo contra una referencia común.

In [ ]:
from transformers import pipeline

# El modelo grande (joeddav/xlm-roberta-large-xnli) ronda los 2.2 GB y es lento en laptop.
# El de Recognai es mucho mas liviano y es el otro que autoriza el enunciado.
MODELO_ZS = "Recognai/bert-base-spanish-wwm-cased-xnli"
# MODELO_ZS = "joeddav/xlm-roberta-large-xnli"   # alternativa si tienes VRAM de sobra

HIPOTESIS = "Esta noticia trata sobre {}."
ETIQUETAS_ES = {"politica": "politica", "economia": "economia",
                "deportes": "deportes", "ciencia_tecnologia": "ciencia y tecnologia"}
INV_ES = {v: k for k, v in ETIQUETAS_ES.items()}

indices = []
for _cat, grupo in df.groupby("etiqueta"):
    indices += list(grupo.sample(min(10, len(grupo)), random_state=SEED).index)
muestra_zs = df.loc[indices].reset_index(drop=True)
print("Muestra zero-shot:", len(muestra_zs), "titulares")
display(muestra_zs["etiqueta"].value_counts())

zs = pipeline("zero-shot-classification", model=MODELO_ZS, device=DEVICE_ID)

t0 = time.time()
pred_zs = []
for texto in muestra_zs["texto"]:
    r = zs(texto, candidate_labels=list(ETIQUETAS_ES.values()),
           hypothesis_template=HIPOTESIS, multi_label=False)
    pred_zs.append(INV_ES[r["labels"][0]])
dur_zs = time.time() - t0

muestra_zs["pred_zs"] = pred_zs
print(f"Tiempo: {dur_zs:.1f}s ({dur_zs/len(muestra_zs):.2f}s por titular)")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

ACC_ZS = accuracy_score(muestra_zs["etiqueta"], muestra_zs["pred_zs"])
F1_ZS  = f1_score(muestra_zs["etiqueta"], muestra_zs["pred_zs"],
                  average="macro", labels=CATEGORIAS, zero_division=0)

print(f"ZERO-SHOT  ({MODELO_ZS})")
print(f"  Accuracy : {ACC_ZS:.3f}")
print(f"  F1 macro : {F1_ZS:.3f}   <- este es el piso minimo a superar\n")
print(classification_report(muestra_zs["etiqueta"], muestra_zs["pred_zs"],
                            labels=CATEGORIAS, zero_division=0))

## 2 · Supervisión débil: etiquetado con un LLM local (Ollama)

Todo corre en la máquina, sin API ni costo por consulta. Se le pide al modelo una respuesta
en **JSON estructurado** con la categoría y su confianza; `format="json"` de Ollama fuerza
que la salida sea JSON válido y evita tener que parsear texto libre.

**Antes de ejecutar:** `ollama serve` tiene que estar corriendo y el modelo descargado
(`ollama pull qwen2.5:7b`). La celda siguiente detecta qué modelos tienes.

In [ ]:
OLLAMA = os.getenv("OLLAMA_HOST", "http://localhost:11434")

def modelos_ollama():
    try:
        r = requests.get(f"{OLLAMA}/api/tags", timeout=10)
        return [m["name"] for m in r.json().get("models", [])]
    except requests.RequestException as e:
        print(f"No se pudo contactar a Ollama en {OLLAMA}: {e}")
        print("Levantalo con:  ollama serve     y descarga un modelo:  ollama pull qwen2.5:7b")
        return []

disponibles = modelos_ollama()
print("Modelos disponibles:", disponibles or "(ninguno)")

PREFERIDOS = ["qwen2.5:7b", "qwen2.5", "qwen3", "llama3.1:8b", "llama3.1", "llama3", "gemma2", "mistral"]
MODELO_LLM = next((m for p in PREFERIDOS for m in disponibles if m.startswith(p)),
                  disponibles[0] if disponibles else "qwen2.5:7b")
print("Modelo elegido para el etiquetado:", MODELO_LLM)

In [ ]:
PROMPT = """Eres un editor de un diario peruano. Clasifica la noticia en EXACTAMENTE una
de estas cuatro categorias:

- politica: gobierno, Congreso, elecciones, partidos, justicia, corrupcion, politica exterior
- economia: mercados, empleo, inflacion, empresas, comercio, finanzas publicas, mineria
- deportes: futbol, seleccion, clubes, atletas, torneos, resultados deportivos
- ciencia_tecnologia: investigacion cientifica, salud, espacio, ambiente, software, IA, gadgets

Responde SOLO con un JSON: {{"categoria": "...", "confianza": 0.0}}
La confianza va de 0 a 1 y refleja que tan seguro estas.

TITULAR: {titular}
BAJADA: {bajada}"""

def etiquetar(titular, bajada, modelo=None, reintentos=1):
    modelo = modelo or MODELO_LLM
    cuerpo = {
        "model": modelo,
        "prompt": PROMPT.format(titular=titular, bajada=(bajada or "")[:400]),
        "format": "json",
        "stream": False,
        "options": {"temperature": 0, "seed": SEED, "num_predict": 80},
    }
    for _ in range(reintentos + 1):
        try:
            r = requests.post(f"{OLLAMA}/api/generate", json=cuerpo, timeout=180)
            datos = json.loads(r.json()["response"])
            cat = str(datos.get("categoria", "")).strip().lower().replace(" ", "_")
            if cat in CATEGORIAS:
                conf = float(datos.get("confianza", 0.5))
                return cat, min(max(conf, 0.0), 1.0)
        except Exception:
            continue
    return None, 0.0

In [ ]:
# Etiquetado del dataset completo, con guardado incremental para poder reanudar
RUTA_LLM = DATOS / "etiquetas_llm.csv"
hechas = {}
if RUTA_LLM.exists():
    previo = pd.read_csv(RUTA_LLM)
    hechas = dict(zip(previo["titular"], zip(previo["etiqueta_llm"], previo["confianza"])))
    print(f"Reanudando: ya habia {len(hechas)} titulares etiquetados")

filas, fallos, t0 = [], 0, time.time()
for i, fila in df.iterrows():
    if fila["titular"] in hechas:
        cat, conf = hechas[fila["titular"]]
    else:
        cat, conf = etiquetar(fila["titular"], fila["bajada"])
        if cat is None:
            fallos += 1
    filas.append({"titular": fila["titular"], "etiqueta_llm": cat, "confianza": conf,
                  "modelo_llm": MODELO_LLM, "fecha": datetime.now().strftime("%Y-%m-%d")})
    if (i + 1) % 25 == 0:
        pd.DataFrame(filas).to_csv(RUTA_LLM, index=False, encoding="utf-8")
        print(f"  {i+1}/{len(df)}  ({time.time()-t0:.0f}s)")

llm = pd.DataFrame(filas)
llm.to_csv(RUTA_LLM, index=False, encoding="utf-8")
print(f"\nListo en {time.time()-t0:.0f}s. Sin respuesta valida: {fallos}")
display(llm["etiqueta_llm"].value_counts(dropna=False))

In [ ]:
# Union con el dataset y descarte de las filas que el LLM no pudo clasificar
df = df.merge(llm[["titular", "etiqueta_llm", "confianza"]], on="titular", how="left")
antes = len(df)
df = df.dropna(subset=["etiqueta_llm"]).reset_index(drop=True)
print(f"Filas descartadas por no tener etiqueta del LLM: {antes - len(df)}")
print(f"Filas para entrenar: {len(df)}\n")

# Acuerdo entre la etiqueta del LLM y la seccion del diario
acuerdo = (df["etiqueta_llm"] == df["etiqueta"]).mean()
print(f"Acuerdo LLM vs. seccion del diario: {acuerdo:.1%}")
print("\nDonde discrepan (filas = seccion, columnas = LLM):")
display(pd.crosstab(df["etiqueta"], df["etiqueta_llm"]))

## 2b · Auditoría humana de 20 casos

Esta parte **la haces tú a mano**: la celda imprime 20 predicciones al azar y abre un CSV
para que marques cada una. No la delegues, es justamente lo que evalúa la rúbrica.

In [ ]:
RUTA_AUD = DATOS / "auditoria_humana.csv"

aud = df.sample(20, random_state=SEED)[
    ["titular", "bajada", "medio", "etiqueta", "etiqueta_llm", "confianza"]].reset_index(drop=True)
aud["de_acuerdo"] = ""       # <- escribe 1 si el LLM acerto, 0 si no
aud["comentario"] = ""

if not RUTA_AUD.exists():
    aud.to_csv(RUTA_AUD, index=False, encoding="utf-8")
    print(f"Creado {RUTA_AUD}. Abrelo, llena la columna 'de_acuerdo' con 1 o 0, "
          f"guarda y vuelve a correr la celda siguiente.\n")

for i, f in aud.iterrows():
    print(f"[{i+1:2d}] LLM={f['etiqueta_llm']:20s} seccion={f['etiqueta']:20s} conf={f['confianza']:.2f}")
    print(f"     {f['titular'][:110]}")

In [ ]:
# Correr DESPUES de llenar la columna 'de_acuerdo' en datos/auditoria_humana.csv
revisada = pd.read_csv(RUTA_AUD)
marcadas = revisada["de_acuerdo"].notna().sum()

if marcadas < 20:
    AUD_OK, AUD_TOTAL = None, 20
    print(f"Faltan por marcar {20 - marcadas} casos en {RUTA_AUD}")
else:
    AUD_OK    = int(revisada["de_acuerdo"].astype(int).sum())
    AUD_TOTAL = len(revisada)
    print(f"AUDITORIA HUMANA: {AUD_OK}/{AUD_TOTAL} = {AUD_OK/AUD_TOTAL:.0%} de acuerdo\n")
    desacuerdos = revisada[revisada["de_acuerdo"].astype(int) == 0]
    if len(desacuerdos):
        print("Casos donde el LLM se equivoco (sirven para la reflexion del README):")
        for _, f in desacuerdos.iterrows():
            print(f"  - [{f['etiqueta_llm']}] {f['titular'][:95]}")
            if isinstance(f.get("comentario"), str) and f["comentario"].strip():
                print(f"      nota: {f['comentario']}")

## 3 · Partición 70 / 15 / 15 estratificada

Semilla fija en 42. **El conjunto de test queda aislado hasta el paso 5**: no se mira, no
se usa para elegir hiperparámetros, no se toca.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.index.values
y = df["etiqueta_llm"].values

idx_tr, idx_tmp = train_test_split(X, test_size=0.30, stratify=y, random_state=SEED)
idx_va, idx_te  = train_test_split(idx_tmp, test_size=0.50,
                                   stratify=df.loc[idx_tmp, "etiqueta_llm"], random_state=SEED)

df["particion"] = "train"
df.loc[idx_va, "particion"] = "val"
df.loc[idx_te, "particion"] = "test"
df[["titular", "medio", "etiqueta", "etiqueta_llm", "particion"]].to_csv(
    DATOS / "particion.csv", index=False, encoding="utf-8")

print("Tamanos:", df["particion"].value_counts().to_dict())
print("\nEstratificacion (proporcion por particion):")
display(pd.crosstab(df["particion"], df["etiqueta_llm"], normalize="index").round(3))

## 4 · Fine-tuning del Transformer

`distilbert-base-multilingual-cased` por defecto: entra en cualquier GPU modesta y el
enunciado lo recomienda. Si tienes VRAM de sobra, cambia a BETO en la línea de abajo.

Configuración mínima del enunciado: 3 épocas, batch 16, evaluación por época, métrica de
optimización **F1 macro**.

In [ ]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

MODELO_BASE = "distilbert-base-multilingual-cased"
# MODELO_BASE = "dccuchile/bert-base-spanish-wwm-cased"   # BETO, mejor en espanol pero mas pesado

tok = AutoTokenizer.from_pretrained(MODELO_BASE)

def a_dataset(parte):
    sub = df[df["particion"] == parte]
    d = Dataset.from_dict({"text": sub["texto"].tolist(),
                           "labels": [LAB2ID[e] for e in sub["etiqueta_llm"]]})
    return d.map(lambda b: tok(b["text"], truncation=True, max_length=96), batched=True)

ds = {p: a_dataset(p) for p in ["train", "val", "test"]}
print({p: len(d) for p, d in ds.items()})

modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE, num_labels=len(CATEGORIAS), id2label=ID2LAB, label2id=LAB2ID)

def metricas(pred):
    logits, y = pred
    yhat = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(y, yhat),
            "f1_macro": f1_score(y, yhat, average="macro", zero_division=0)}

In [ ]:
import inspect

opciones = dict(
    output_dir=str(MODELO / "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=20,
    seed=SEED,
    report_to="none",
    fp16=(DEVICE == "cuda"),       # mitad de precision solo tiene sentido en CUDA
)

# El argumento cambio de nombre en transformers 4.41: antes era evaluation_strategy.
# Se elige el que acepte la version instalada, para que el notebook corra en ambas.
acepta = set(inspect.signature(TrainingArguments.__init__).parameters)
opciones["eval_strategy" if "eval_strategy" in acepta else "evaluation_strategy"] = "epoch"
if "use_cpu" in acepta:
    opciones["use_cpu"] = (DEVICE == "cpu")

args = TrainingArguments(**opciones)
print("Entrenando sobre:", DEVICE, "| fp16:", opciones["fp16"])

entrenador = Trainer(model=modelo, args=args,
                     train_dataset=ds["train"], eval_dataset=ds["val"],
                     data_collator=DataCollatorWithPadding(tok), compute_metrics=metricas)

t0 = time.time()
entrenador.train()
DUR_ENTRENAMIENTO = time.time() - t0
print(f"\nEntrenamiento terminado en {DUR_ENTRENAMIENTO/60:.1f} min sobre {DEVICE}")

modelo.save_pretrained(MODELO); tok.save_pretrained(MODELO)
print("Modelo guardado en modelo/")

## 5 · Evaluación final sobre el test reservado

Recién aquí se toca el test. Se reporta accuracy, **F1 macro** y la matriz de confusión.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

salida = entrenador.predict(ds["test"])
y_true = np.array(salida.label_ids)
y_pred = np.argmax(salida.predictions, axis=-1)

ACC_FT = accuracy_score(y_true, y_pred)
F1_FT  = f1_score(y_true, y_pred, average="macro", zero_division=0)

print(f"TEST — Transformer fine-tuned ({MODELO_BASE})")
print(f"  Accuracy : {ACC_FT:.3f}")
print(f"  F1 macro : {F1_FT:.3f}\n")
print(classification_report(y_true, y_pred, target_names=CATEGORIAS, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6.5, 5.2))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=CATEGORIAS, yticklabels=CATEGORIAS, ax=ax)
ax.set_xlabel("Predicho"); ax.set_ylabel("Real (etiqueta del LLM)")
ax.set_title(f"Matriz de confusion en test — F1 macro {F1_FT:.3f}")
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig(DATOS / "figs" / "matriz_confusion.png", dpi=150)
plt.show()

In [ ]:
# Control externo: el mismo test, pero contra la seccion del diario,
# que NUNCA participo del entrenamiento
test = df[df["particion"] == "test"].reset_index(drop=True)
test["pred"] = [ID2LAB[i] for i in y_pred]

F1_FT_SECCION  = f1_score(test["etiqueta"], test["pred"], average="macro",
                          labels=CATEGORIAS, zero_division=0)
ACC_FT_SECCION = accuracy_score(test["etiqueta"], test["pred"])

print("CONTROL EXTERNO — mismo test, referencia = seccion del diario")
print(f"  Accuracy : {ACC_FT_SECCION:.3f}")
print(f"  F1 macro : {F1_FT_SECCION:.3f}")
print(f"\nBrecha frente a la referencia del LLM: {F1_FT - F1_FT_SECCION:+.3f}")

print("\nDesempeno por categoria (referencia = LLM):")
por_cat = pd.DataFrame({
    "soporte": [int((y_true == i).sum()) for i in range(len(CATEGORIAS))],
    "f1": f1_score(y_true, y_pred, average=None, zero_division=0,
                   labels=list(range(len(CATEGORIAS)))),
}, index=CATEGORIAS).sort_values("f1")
display(por_cat.round(3))
print(f"Mejor categoria : {por_cat.index[-1]}")
print(f"Peor categoria  : {por_cat.index[0]}")

### Reflexión por categoría

*Mira la matriz y la tabla de arriba y escribe aquí tu lectura. Pistas de qué suele pasar
con titulares peruanos — confirma o descarta cada una con tus números:*

- **`deportes`** suele ser la más fácil: vocabulario cerrado y propio (nombres de clubes,
  marcadores, "venció", "clasificó") que no aparece en ninguna otra categoría.
- **`politica` ↔ `economia`** es la confusión clásica: presupuesto, retiro de AFP, impuestos
  o minería son a la vez decisión política y hecho económico. Mira si la matriz lo confirma.
- **`ciencia_tecnologia`** arrastra el ruido de las secciones de origen: El Comercio publica
  bajo `tecnologia` notas de salud y ambiente, y La República mete bajo `ciencia` divulgación
  internacional. Es la categoría más heterogénea del corpus.

**Tu lectura:** …

## 6 · Comparativa y decisión

Los tres métodos medidos sobre **el mismo conjunto de test** y la misma referencia, para que
la comparación sea justa.

In [ ]:
# Zero-shot sobre el test completo, para comparar contra el modelo fine-tuned
t0 = time.time()
pred_zs_test = []
for texto in test["texto"]:
    r = zs(texto, candidate_labels=list(ETIQUETAS_ES.values()),
           hypothesis_template=HIPOTESIS, multi_label=False)
    pred_zs_test.append(INV_ES[r["labels"][0]])
DUR_ZS_TEST = time.time() - t0

test["pred_zs"] = pred_zs_test
F1_ZS_TEST = f1_score(test["etiqueta_llm"], test["pred_zs"], average="macro",
                      labels=CATEGORIAS, zero_division=0)

# El "LLM debil" sobre el test es, por construccion, la referencia: F1 = 1.0 contra si mismo.
# Lo honesto es medirlo contra la seccion del diario.
F1_LLM_SECCION = f1_score(test["etiqueta"], test["etiqueta_llm"], average="macro",
                          labels=CATEGORIAS, zero_division=0)

comparativa = pd.DataFrame([
    {"metodo": "Zero-shot (sin entrenar)", "F1 macro (ref. LLM)": round(F1_ZS_TEST, 3),
     "F1 macro (ref. seccion)": round(f1_score(test["etiqueta"], test["pred_zs"], average="macro",
                                               labels=CATEGORIAS, zero_division=0), 3),
     "seg/titular": round(DUR_ZS_TEST/len(test), 2), "costo": "gratis, local"},
    {"metodo": f"LLM debil ({MODELO_LLM})", "F1 macro (ref. LLM)": 1.000,
     "F1 macro (ref. seccion)": round(F1_LLM_SECCION, 3),
     "seg/titular": None, "costo": "gratis, local, pero lento"},
    {"metodo": f"Transformer fine-tuned", "F1 macro (ref. LLM)": round(F1_FT, 3),
     "F1 macro (ref. seccion)": round(F1_FT_SECCION, 3),
     "seg/titular": None, "costo": "entrenar una vez, luego milisegundos"},
])
display(comparativa)
print("\nNota: el LLM debil marca 1.000 contra si mismo por construccion, no porque sea perfecto.")

In [ ]:
# Latencia de inferencia del modelo fine-tuned, que es su argumento mas fuerte
from transformers import pipeline as pl
clf = pl("text-classification", model=str(MODELO), tokenizer=str(MODELO),
         device=DEVICE_ID, truncation=True, max_length=96)
_ = clf("calentamiento")
t0 = time.time()
for t in test["texto"].head(50):
    clf(t)
LAT_FT = (time.time() - t0) / 50
print(f"Latencia del Transformer: {LAT_FT*1000:.1f} ms por titular")
print(f"Zero-shot, en comparacion: {DUR_ZS_TEST/len(test)*1000:.0f} ms por titular")
print(f"-> el modelo entrenado es ~{(DUR_ZS_TEST/len(test))/LAT_FT:.0f}x mas rapido")

### ¿Cuál elegirías para producción?

*Escribe tu justificación con tus números. El argumento tiene tres patas:*

- **Precisión:** compara F1 del fine-tuned contra el piso zero-shot.
- **Costo computacional:** el LLM de 7B pide varios GB de VRAM y segundos por titular;
  DistilBERT son ~135M de parámetros y corre hasta en CPU.
- **Latencia:** si el clasificador va detrás de una API que recibe titulares en vivo, la
  diferencia entre milisegundos y segundos decide si el sistema es usable.

*La respuesta esperable: el LLM se usa **una vez**, para fabricar las etiquetas; el
Transformer es el que se despliega. Es exactamente el patrón que el profesor mostró en la
clase de OCR — el trabajo masivo y barato en local, el modelo caro solo donde agrega valor.*

**Tu decisión:** …

## 7 · Guardar métricas para el dashboard

In [ ]:
METRICAS = {
    "f1_macro_test": float(F1_FT),
    "accuracy_test": float(ACC_FT),
    "f1_macro_test_vs_seccion": float(F1_FT_SECCION),
    "f1_zero_shot_test": float(F1_ZS_TEST),
    "f1_zero_shot_muestra40": float(F1_ZS),
    "accuracy_zero_shot_muestra40": float(ACC_ZS),
    "auditoria_acuerdo": AUD_OK,
    "auditoria_total": AUD_TOTAL,
    "acuerdo_llm_seccion": float(acuerdo),
    "modelo_base": MODELO_BASE,
    "modelo_llm": MODELO_LLM,
    "modelo_zero_shot": MODELO_ZS,
    "dispositivo": DEVICE,
    "minutos_entrenamiento": round(DUR_ENTRENAMIENTO / 60, 1),
    "latencia_ms": round(LAT_FT * 1000, 1),
    "n_total": int(len(df)),
    "n_train": int((df["particion"] == "train").sum()),
    "n_val": int((df["particion"] == "val").sum()),
    "n_test": int((df["particion"] == "test").sum()),
    "fecha": datetime.now().strftime("%Y-%m-%d"),
}
(MODELO / "metricas.json").write_text(json.dumps(METRICAS, indent=2, ensure_ascii=False),
                                      encoding="utf-8")
print(json.dumps(METRICAS, indent=2, ensure_ascii=False))

## 8 · Línea para el comentario de entrega

Copia la salida de la celda siguiente al comentario del Issue #188.

In [ ]:
print("Tarea 1 – Individual")
print("Medio: Comparada")
print("Estudiante: Alexander Noblejas (@TU_USUARIO_GITHUB)")
print("Dashboard en vivo: TU_ENLACE  (o: ejecucion local comprobada)")
print("Carpeta del trabajo: https://github.com/TU_USUARIO/TU_REPO/tree/main/tarea_1")
print(f"Resultados Test: F1 Macro: {F1_FT:.2f} | Zero-shot: {F1_ZS_TEST:.2f} | "
      f"Acuerdo LLM-Humano: {AUD_OK}/{AUD_TOTAL}")

## 9 · (Opcional) Publicar el modelo en Hugging Face Hub

Los pesos pesan más de 100 MB y están en `.gitignore`, así que **Streamlit Cloud no los va a
ver en GitHub**. Si vas a desplegar, súbelos al Hub y define `HF_MODEL_ID` como secreto en
Streamlit. El token va en la variable de entorno `HF_TOKEN`, nunca en el repo.

Si solo vas a demostrar ejecución local, sáltate esta celda.

In [ ]:
SUBIR_AL_HUB = False
REPO_HUB = "TU_USUARIO_HF/noticias-pe-distilbert"

if SUBIR_AL_HUB:
    tokhf = os.environ["HF_TOKEN"]
    modelo.push_to_hub(REPO_HUB, token=tokhf)
    tok.push_to_hub(REPO_HUB, token=tokhf)
    print("Subido. Define HF_MODEL_ID =", REPO_HUB)
else:
    print("Omitido. El dashboard usara el modelo local en modelo/.")